---
title: Aplanatic Aberrations Notebook
authors: [gvarnavides]
date: 2026-05-28
---

In [ ]:
%matplotlib widget

import abtem
import numpy as np
import quantem as em
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib.gridspec import GridSpec
import ipywidgets

plt.rcParams['text.color']='white'
plt.rcParams['xtick.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.labelcolor'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'white'
# plt.rcParams.update({
#     "text.usetex": True,
#     "text.latex.preamble": r"\usepackage{amsmath}"
# })

abtem.config.set({"dask.lazy":False});


In [2]:
def array_to_scaled_rgba(array,vmin=0.02,vmax=0.98):
    if np.iscomplexobj(array):
        scaled_amplitude = np.abs(array)
        scaled_angle = np.angle(array)
    else:
        scaled_amplitude = array
        scaled_angle = None

    if scaled_amplitude.std() > 1e-12:
        vmin, vmax = np.quantile(scaled_amplitude,(vmin,vmax))

        scaled_amplitude = (scaled_amplitude.clip(vmin,vmax) - vmin) / (vmax-vmin)
    else:
        scaled_amplitude = np.ones_like(scaled_amplitude)

    rgba = em.visualization.visualization_utils.array_to_rgba(
        scaled_amplitude,
        scaled_angle
    )

    return rgba

def return_probe_arrays(semiangle_cutoff,aberrations=None,**kwargs):
    """ """
    ctf = abtem.CTF(
        semiangle_cutoff=semiangle_cutoff,
        sampling=(0.125,0.125),
        gpts=(128,128),
        energy=300e3,
        aberration_coefficients=aberrations,
        **kwargs,
    )
    
    fourier_probe = ctf.to_diffraction_patterns(
        gpts=ctf.gpts
    ).array
    real_probe = ctf.to_point_spread_functions(
        gpts=ctf.gpts,
        extent=ctf.extent
    ).array
    
    return ctf, fourier_probe, real_probe    

### Aberration Function

In [3]:
ab_ctf, ab_fourier_probe, ab_real_probe  = return_probe_arrays(
    semiangle_cutoff=20,
    defocus=100,
    astigmatism=50,
    astigmatism_angle=np.pi/4,
)

width = 620
aspect_ratio = 0.45
height = int(width * aspect_ratio)

dpi = 72
with plt.ioff():
    ab_fig,ab_axs = plt.subplots(1,2,figsize=(width/dpi,height/dpi),dpi=dpi)

ab_rgb_fourier_probe = array_to_scaled_rgba(ab_fourier_probe,vmin=0.001,vmax=0.999)
ab_im_fourier = ab_axs[0].imshow(ab_rgb_fourier_probe)

ab_rgb_real_probe = array_to_scaled_rgba(ab_real_probe,vmin=0.001,vmax=0.999)
ab_im_real = ab_axs[1].imshow(ab_rgb_real_probe)

scalebar_real = em.visualization.ScalebarConfig(ab_ctf.sampling[0],units=r'$\AA$')
scalebar_fourier = em.visualization.ScalebarConfig(ab_ctf.angular_sampling[0],units='mrad')

bars = [scalebar_fourier, scalebar_real]
titles= ["Fourier-space probe", "real-space probe"]

for ax, bar, title in zip(ab_axs, bars, titles):
    ax.patch.set_alpha(0)
    ax.set(xticks=[],yticks=[],title=title)
    divider = make_axes_locatable(ax)
    ax_cb = divider.append_axes("right", size="5%", pad="2.5%")
    em.visualization.visualization_utils.add_arg_cbar_to_ax(ab_fig,ax_cb)
    em.visualization.visualization_utils.add_scalebar_to_ax(
        ax,
        ab_rgb_fourier_probe.shape[1],
        bar.sampling,
        bar.length,
        bar.units,
        bar.width_px,
        bar.pad_px,
        bar.color,
        bar.loc,
    )


ab_fig.tight_layout()
ab_fig.canvas.resizable = False
ab_fig.canvas.header_visible = False
ab_fig.canvas.footer_visible = False
ab_fig.canvas.toolbar_visible = False
ab_fig.canvas.layout.width = f'{width}px'
ab_fig.canvas.toolbar_position = 'bottom'
ab_fig.patch.set_alpha(0)
None

In [4]:
def update_aberrated_plot(
    semiangle_cutoff,
    defocus_nm,
    astigmatism_nm,
    astigmatism_angle_deg,
    coma_um,
    coma_angle_deg,
    ):
    """ """
    ab_ctf, ab_fourier_probe, ab_real_probe  = return_probe_arrays(
        semiangle_cutoff=semiangle_cutoff,
        defocus=defocus_nm*10, # nm -> A
        astigmatism=astigmatism_nm*10, # nm -> A
        astigmatism_angle=np.deg2rad(astigmatism_angle_deg),
        coma = coma_um*1e4, # um -> A 
        coma_angle = np.deg2rad(coma_angle_deg),
    )

    ab_rgb_fourier_probe = array_to_scaled_rgba(ab_fourier_probe,vmin=0.001,vmax=0.999)
    ab_im_fourier.set_data(ab_rgb_fourier_probe)
    

    ab_rgb_real_probe = array_to_scaled_rgba(ab_real_probe,vmin=0.001,vmax=0.999)
    ab_im_real.set_data(ab_rgb_real_probe)
    ab_fig.canvas.draw_idle()
    return None

style = {
    'description_width': 'initial',
}

ab_layout = ipywidgets.Layout(width=f'{width//2}px',height='30px')

ab_semiangle_slider = ipywidgets.FloatSlider(
    min=5,
    max=50,
    step=0.5,
    value=20,
    layout=ab_layout,
    style=style,
    description="semi-angle [mrad]",
)

ab_defocus_slider = ipywidgets.FloatSlider(
    min=-15,
    max=15,
    step=0.5,
    value=10,
    layout=ab_layout,
    style=style,
    description="defocus [nm]",
)

ab_astigmatism_slider = ipywidgets.FloatSlider(
    min=0,
    max=10,
    step=0.1,
    value=5,
    layout=ab_layout,
    style=style,
    description="astigmatism [nm]",
)

ab_coma_slider = ipywidgets.FloatSlider(
    min=0,
    max=2,
    step=0.05,
    value=0,
    layout=ab_layout,
    style=style,
    description="coma [μm]",
)

ab_astigmatism_angle_slider = ipywidgets.FloatSlider(
    min=-90,
    max=90,
    step=1,
    value=45,
    layout=ab_layout,
    style=style,
    description="astigmatism angle [°]",
)

ab_coma_angle_slider = ipywidgets.FloatSlider(
    min=-180,
    max=180,
    step=1,
    value=180,
    layout=ab_layout,
    style=style,
    description="coma angle [°]",
)

ipywidgets.interactive_output(
    update_aberrated_plot,
    {
        'semiangle_cutoff': ab_semiangle_slider,
        'defocus_nm': ab_defocus_slider,
        'astigmatism_nm': ab_astigmatism_slider,
        'astigmatism_angle_deg': ab_astigmatism_angle_slider,
        'coma_um': ab_coma_slider,
        'coma_angle_deg': ab_coma_angle_slider
    }
)
None

In [5]:
#| label: app:aberrations_widget
display(
    ipywidgets.VBox(
        [
            ipywidgets.HBox([ab_semiangle_slider,ab_defocus_slider]),
            ipywidgets.HBox([ab_astigmatism_slider,ab_astigmatism_angle_slider]),
            ipywidgets.HBox([ab_coma_slider,ab_coma_angle_slider]),
            ab_fig.canvas,
        ],
        layout=ipywidgets.Layout(
            align_items="center"
        )
    )
)